Declustering algorytmem Gardnera-Knopoffa, na wzór metody użytej w opragromowaniu OpenQuake.

In [ ]:


import os
from pathlib import Path
import numpy as np
import pandas as pd

home_path = Path.home()
input_dir = home_path / .../ "dane"
input_file = os.path.join(input_dir, "eq_data_plus65_2.csv") #przefiltrowane dane w tym przypadku M>6.5
output_file = os.path.join(input_dir, "eq_data_declustered_m65_2.csv")


# parametr słuący do usuwania foreshockow, wziete z openquake.
FS_TIME_PROP = 1.0


#  Odległość na sferze

def haversine_distance(lat1, lon1, lat2, lon2):
    R_earth = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R_earth * c


# okno czasowo-przestrzenne Gardnera-Knopoffa
def get_gk_window(magnitude):

    magnitude = np.asarray(magnitude, dtype=float)
    r_km = 10 ** (0.1238 * magnitude + 0.983)

    t_days = np.where(
        magnitude >= 6.5,
        10 ** (0.032 * magnitude + 2.7389),
        10 ** (0.5409 * magnitude - 0.547),
    )
    return r_km, t_days


# Wczytanie przefiltrowanych danych
df = pd.read_csv(input_file)
df["time"] = pd.to_datetime(df["time"])

# sortowanie od największej magnitudy do najmniejszej, aby najpierw oznaczać wstrząsy główne
df_gk = df.sort_values(by="mag", ascending=False).copy()
df_gk["is_aftershock"] = False

total_records = len(df_gk)
#pętla po wszystkich rekordach, aby oznaczyć aftershocki i foreshocki
for i in range(total_records):
    if df_gk.iloc[i]["is_aftershock"]:
        continue

    main_shock = df_gk.iloc[i]

    r_max, t_max = get_gk_window(main_shock["mag"])

    # Różnica czasu:]]
    time_diffs = (df_gk["time"] - main_shock["time"]).dt.total_seconds() / (24 * 3600)

    # okno przed i po wstrząsie
    time_mask = (time_diffs > -t_max * FS_TIME_PROP) & (time_diffs <= t_max) & (time_diffs != 0)

    if not time_mask.any():
        continue

    candidates = df_gk[time_mask]
    distances = haversine_distance(
        main_shock["lat"],
        main_shock["long"],
        candidates["lat"],
        candidates["long"],
    )


    linked_indices = candidates[distances <= r_max].index
    df_gk.loc[linked_indices, "is_aftershock"] = True

# Filtrowanie
df_clean = df_gk[df_gk["is_aftershock"] == False].copy()
df_clean = df_clean.sort_values(by="time").drop(columns=["is_aftershock"])

df_clean.to_csv(output_file, index=False)

print(f"Liczba wejściowa trzęsień: {len(df)}")
print(f"Liczba niezależnych wstrząsów głównych: {len(df_clean)}")



TypeError: unsupported operand type(s) for /: 'PosixPath' and 'ellipsis'